In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

In [66]:
import torch
from torch.utils.data import Dataset

import random
import numpy as np
import pandas as pd
import os
import csv

from data_utils import tokenize

class ContraSeqDataset(Dataset):
    
    def __init__(self, anc_path, aug_path,
             start_token='<', 
             end_token='>', 
             pad_token = 'X',
             property_pred = None,
             max_len = 120,
             use_cuda=None):
        
        super().__init__() 
        
        self.anc_smi = np.transpose(pd.read_csv(anc_path)['smiles'].values)
        self.aug_smi = np.asarray(pd.read_csv(aug_path))
        
        augs_arr = np.asarray([x[1] for x in self.aug_smi])
        tokens,_ = tokenize( np.hstack((self.anc_smi,augs_arr)) )  
        all_tokens = list(set(tokens + start_token + end_token + pad_token))
        self.tokens = ''.join(list(np.sort(all_tokens)))        
               
        self.start_token = start_token
        self.end_token = end_token
        self.pad = pad_token

        self.max_sm_len = max_len
        self.max_len = max_len + 2
        
        self.property_pred = property_pred
    
    def idc_tensor(self, smi):
        tensor = torch.zeros(len(smi)).long()
        for i in range(len(smi)):
            tensor[i] = self.tokens.index(smi[i])
        return tensor
    
    def get_vec(self, smi):
        padding = ''.join([self.pad for _ in range(self.max_sm_len - len(smi))])
        smi = self.start_token + smi + self.end_token + padding
        vec = self.idc_tensor(smi)
        return vec
 

    def remove_extra_tokens(self, smi):
        smi = smi.replace(self.pad,'')
        smi = smi.replace(self.start_token,'')
        return smi.replace(self.end_token,'')
    
    def convert_vec_to_smi(self, vec, snip=False):
        smi_arr = np.array(list(self.tokens))[vec.cpu().detach().numpy()]
        smi_list = [''.join(arr) for arr in smi_arr]
        if snip:
            smi_list = [self.remove_extra_tokens(smi) for smi in smi_list]
        return smi_list
        
    def __len__(self):
        return self.data_len
    
    def __getitem__(self,idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
            
        anc_vec = self.get_vec(self.anc_smi[idx])        
        aug_smileses = self.aug_smi[self.aug_smi[:,0]==idx][:,1]
        aug_vecs = torch.stack([self.get_vec(sm) for sm in aug_smileses])
        
        sample = {'anchor': anc_vec,
                  'augs': aug_vecs}
        
        return sample

In [67]:
anc_path = 'model_dataset/anchor_smiles.csv'
aug_path = 'model_dataset/augmented_smiles.csv'

dataset = ContraSeqDataset(anc_path, aug_path)
samp = dataset.__getitem__(666)
# print(samp['anchor'])

augs = dataset.convert_vec_to_smi(samp['augs'], snip=True)
print(augs)
# print(samp['augs'])

['O=C1NC(c2ccccc2F)SC1O', 'Cc1cccc(F)c1C1NC(=O)CS1', 'CC1(c2ccccc2F)NC(=O)CS1', 'Cc1ccc(C2NC(=O)CS2)c(F)c1', 'Cc1ccc(F)c(C2NC(=O)CS2)c1', 'Cc1cccc(C2NC(=O)CS2)c1F', 'O=C1C[SH](O)C(c2ccccc2F)N1', 'CN1C(=O)CSC1c1ccccc1F']


In [45]:
test = np.array([11, 20, 12, 14,  5, 14, 23, 14,  1, 27,  6, 27, 27, 27, 27, 27,  6, 15,
         2, 19,  5, 13, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24])
tok = np.array(list(tokens))

In [46]:
tok[test]

array(['<', 'O', '=', 'C', '1', 'C', 'S', 'C', '(', 'c', '2', 'c', 'c',
       'c', 'c', 'c', '2', 'F', ')', 'N', '1', '>', 'X', 'X', 'X', 'X',
       'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X',
       'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X',
       'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X',
       'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X',
       'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X',
       'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X',
       'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X', 'X',
       'X', 'X', 'X', 'X', 'X'], dtype='<U1')

In [3]:
anc_path = 'model_dataset/anchor_smiles.csv'
aug_path = 'model_dataset/augmented_smiles.csv'

anc_smi = np.transpose(pd.read_csv(anc_path)['smiles'].values)
aug_smi = np.asarray(pd.read_csv(aug_path))

In [15]:
from data_utils import tokenize

start_token='<'
end_token='>'
pad_token = 'X'
aug = np.asarray([x[1] for x in aug_smi])
tokens,_ = tokenize( np.hstack((anc_smi,aug)) )
all_tokens = list(set(tokens + start_token + end_token + pad_token))
tokens = ''.join(list(np.sort(all_tokens)))        

In [12]:
def idc_tensor(smi):
    tensor = torch.zeros(len(smi)).long()
    for i in range(len(smi)):
        tensor[i] = tokens.index(smi[i])
    return tensor

def get_vec(smi):
    vec = idc_tensor(smi)
    return vec
    
idx = 666

# print(anc_smi[idx])

anc_vec = get_vec(anc_smi[idx])        
print(anc_vec)

aug_smileses = aug_smi[aug_smi[:,0]==idx][:,1]
aug_vecs = [get_vec(sm) for sm in aug_smileses]
print(aug_vecs)

tensor([18, 11, 12,  5, 12, 21, 12,  1, 24,  6, 24, 24, 24, 24, 24,  6, 13,  2,
        17,  5])
[tensor([18, 11, 12,  5, 17, 12,  1, 24,  6, 24, 24, 24, 24, 24,  6, 13,  2, 21,
        12,  5, 18]), tensor([12, 24,  5, 24, 24, 24, 24,  1, 13,  2, 24,  5, 12,  5, 17, 12,  1, 11,
        18,  2, 12, 21,  5]), tensor([12, 12,  5,  1, 24,  6, 24, 24, 24, 24, 24,  6, 13,  2, 17, 12,  1, 11,
        18,  2, 12, 21,  5]), tensor([12, 24,  5, 24, 24, 24,  1, 12,  6, 17, 12,  1, 11, 18,  2, 12, 21,  6,
         2, 24,  1, 13,  2, 24,  5]), tensor([12, 24,  5, 24, 24, 24,  1, 13,  2, 24,  1, 12,  6, 17, 12,  1, 11, 18,
         2, 12, 21,  6,  2, 24,  5]), tensor([12, 24,  5, 24, 24, 24, 24,  1, 12,  6, 17, 12,  1, 11, 18,  2, 12, 21,
         6,  2, 24,  5, 13]), tensor([18, 11, 12,  5, 12, 22, 21, 14, 23,  1, 18,  2, 12,  1, 24,  6, 24, 24,
        24, 24, 24,  6, 13,  2, 17,  5]), tensor([12, 17,  5, 12,  1, 11, 18,  2, 12, 21, 12,  5, 24,  5, 24, 24, 24, 24,
        24,  5, 13])]


In [6]:
np.argwhere(aug_smi[:,0]==22)

array([[155],
       [156],
       [157],
       [158],
       [159],
       [160]])

In [7]:
aug_smi[aug_smi[:,0]==22][:,1]

array(['NC1CC1c1nc[nH]c1O', 'NC1CC1(O)c1c[nH]cn1', 'CNC1CC1c1c[nH]cn1',
       'CC1(N)CC1c1c[nH]cn1', 'CC1C(N)C1c1c[nH]cn1',
       'Cc1nc(C2CC2N)c[nH]1'], dtype=object)